# CS336 Spring 2026 - Profile and Optimize BPE Training
----
本节不增加新的 tokenizer 语义, 而是第一次进入
```text
Algorithm Optimization + Syetems Thinking
```
本节路线为
```text
reference trainer
      ↓
benchmark
      ↓
cost decomposition
      ↓
merge locality
      ↓
inverted index
      ↓
incremental pair maintenance
      ↓
differential testing
      ↓
benchmark again
```
> 本节先有可信的 oracle, 再优化; 先证明结果相同, 再讨论快多少

## 1. Reference Baseline
上一节内部表示继续保留
```text
Token = bytes

TokenSequence = tuple[bytes, ...]

Pair = tuple[bytes, bytes]
```
训练 corpus state:
```text
PretokenCounts

TokenSequence
        ↓
corpus frequency
```
这一节的 benchmark 直接从 PretokenCounts 开始. 这样可以单独研究
$$
T_{merge}
$$
以后再研究完整
$$
T_{total}=T_{I/O}+T_{merge}+T_{pretoken}
$$

In [1]:
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass
from time import perf_counter
from typing import TypeAlias

import random

Token: TypeAlias = bytes
TokenSequence: TypeAlias = tuple[Token, ...]
PretokenCounts: TypeAlias = Counter[TokenSequence]
Pair: TypeAlias = tuple[Token, Token]

In [2]:
def local_pair_counts(sequence: TokenSequence) -> Counter[Pair]:
    return Counter(zip(sequence, sequence[1:]))


def count_adjacent_pairs(pretoken_counts: PretokenCounts) -> Counter[Pair]:
    result: Counter[Pair] = Counter()

    for sequence, frequency in pretoken_counts.items():
        for pair, local_count in local_pair_counts(sequence).items():
            result[pair] += local_count * frequency

    return result


In [3]:
def select_best_pair(pair_counts: Counter[Pair]) -> Pair | None:

    if not pair_counts:
        return None

    return max(pair_counts, key=lambda pair: (pair_counts[pair], pair))


In [4]:
def merge_pair_in_sequence(sequence: TokenSequence, pair: Pair) -> TokenSequence:
    left, right = pair
    merged = left + right
    output: list[Token] = []

    i = 0
    while i < len(sequence):
        if i + 1 < len(sequence) and sequence[i] == left and sequence[i + 1] == right:
            output.append(merged)
            i += 2

        else:
            output.append(sequence[i])
            i += 1

    return tuple(output)


In [5]:
def apply_merge_reference(
    pretoken_counts: PretokenCounts, pair: Pair
) -> PretokenCounts:
    updated: PretokenCounts = Counter()

    for sequence, frequency in pretoken_counts.items():
        updated[merge_pair_in_sequence(sequence, pair)] += frequency

    return updated


In [6]:
def train_reference(
    initial_counts: PretokenCounts, num_merges: int
) -> tuple[list[Pair], PretokenCounts]:
    counts = Counter(initial_counts)
    merges: list[Pair] = []

    for _ in range(num_merges):
        pair_counts = count_adjacent_pairs(counts)
        best_pair = select_best_pair(pair_counts)

        if best_pair is None:
            break

        merges.append(best_pair)
        counts = apply_merge_reference(counts, best_pair)

    return (merges, counts)


上述就是接下来所有的 optimized implementations 的 Correctness Oracle.

## 2. Naive Trainer 哪里浪费?
考虑第 t 轮 merge:
```text
所有 distinct sequences
        ↓
重新计算全部 adjacent pairs
        ↓
选择 winner
        ↓
再次扫描所有 sequences
        ↓
尝试应用 winner
```
假设 winner 是 `(t, h)`, 在 corpus 中真正包含的只有很少的几个, 但是 naive trainer 会扫描所有 sequences, 重新计算所有 adjacent pairs, 选择 winner, 再次扫描所有 sequences, 这些操作都是不必要的.
> 真的的问题是, 每一轮都重新计算大量没有变化的 state.

## 3. 建立 Benchmark Corpus
对于此问题, 不能只用 `the cat` benchmark. 而是构造
```text
many distinct sequences
+
different corpus frequencies
+
fixed random seed
```
并且直接生成 PretokenCounts, 从而只测 BPE merge loop.

In [7]:
def make_synthetic_counts(
    *,
    num_sequences: int = 900,
    min_length: int = 6,
    max_length: int = 16,
    seed: int = 336,
) -> PretokenCounts:
    rng = random.Random(seed)
    alphabet = [bytes([value]) for value in range(ord("a"), ord("z") + 1)]
    counts: PretokenCounts = Counter()

    while len(counts) < num_sequences:
        length = rng.randint(min_length, max_length)
        sequence = tuple(rng.choice(alphabet) for _ in range(length))
        counts[sequence] += rng.randint(1, 50)

    return counts

In [8]:
benchmark_counts = make_synthetic_counts()

print("distinct sequences:", len(benchmark_counts))
print("weighted occurrences:", sum(benchmark_counts.values()))
print(
    "distinct-state token positions:",
    sum(len(sequence) for sequence in benchmark_counts),
)

distinct sequences: 900
weighted occurrences: 22639
distinct-state token positions: 9883


## 4. Benchmark 应该如何做
单次的
```Python
start = time()
...
end = time()
```
不能作为严谨的 Benchmark. 至少应该控制
```text
same input
same seed
same algorithm semantics
multiple repetitions
median
```
而真实的系统 Benchmark 还要考虑
```text
warm-up
CPU frequency
background processes
cache
memory pressure
```
本节的目标是建立
```text
baseline
↓
change one idea
↓
same correctness
↓
measure again
```

In [9]:
def time_call(fn, *args, repeats: int = 3, **kwargs):
    durations = []
    result = None

    for _ in range(repeats):
        start = perf_counter()
        result = fn(*args, **kwargs)
        durations.append(perf_counter() - start)

    durations.sort()
    median = durations[len(durations) // 2]

    return median, result

In [10]:
NUM_BENCHMARK_MERGES = 40

reference_time, reference_result = time_call(
    train_reference, benchmark_counts, NUM_BENCHMARK_MERGES, repeats=3
)

print("reference median seconds:", round(reference_time, 4))
print("merges learned:", len(reference_result[0]))

reference median seconds: 0.2042
merges learned: 40


## 5. Cost Decomposition
将 reference trainer 拆为三个阶段
```text
pair recount
winner selection
merge application
```
也就是
$$
T_{merge\ loop}=T_{count}+T_{winner}+T_{apply}
$$

In [11]:
@dataclass
class ReferenceProfile:
    pair_count_seconds: float = 0.0
    winner_seconds: float = 0.0
    merge_seconds: float = 0.0

In [12]:
def profile_reference(
    initial_counts: PretokenCounts, num_merges: int
) -> ReferenceProfile:
    counts = Counter(initial_counts)
    profile = ReferenceProfile()

    for _ in range(num_merges):
        start = perf_counter()
        pair_counts = count_adjacent_pairs(counts)
        profile.pair_count_seconds += perf_counter() - start

        start = perf_counter()
        pair = select_best_pair(pair_counts)
        profile.winner_seconds += perf_counter() - start

        if pair is None:
            break

        start = perf_counter()
        counts = apply_merge_reference(counts, pair)
        profile.merge_seconds += perf_counter() - start

    return profile

In [13]:
profile = profile_reference(benchmark_counts, NUM_BENCHMARK_MERGES)
total = profile.pair_count_seconds + profile.winner_seconds + profile.merge_seconds

for name, value in [
    ("pair recount", profile.pair_count_seconds),
    ("winner scan", profile.winner_seconds),
    ("merge all", profile.merge_seconds),
]:
    print(f"{name:12s}: {value:.4f}s ({100 * value / total:5.1f}%)")


pair recount: 0.1253s ( 73.1%)
winner scan : 0.0044s (  2.6%)
merge all   : 0.0418s ( 24.4%)


上述结果表明: 当前 workload 中, pair recount 和 merge all 占据绝大多数时间. 所以下一步的目的是去优化这两个.

## 6. Inverted Index
现在真正需要知道的是:
<center>
winner pair 出现在哪些 distinct sequences 中
</center>

之前一直是 `sequences -> pairs`, 现在反过来要找 `pairs -> sequences`. 定义
$$
I(p)=\{w: p \ occurs \ in \ w\}
$$
于是 winner $p_t^*$ 选出来以后, 只访问 $I(p_t^*)$, 而不是整个 corpus state. 这就是
<center>
Inverted Index
</center>

In [14]:
def build_pair_index(pretoken_counts: PretokenCounts) -> dict[Pair, set[TokenSequence]]:
    index: dict[Pair, set[TokenSequence]] = defaultdict(set)

    for sequence in pretoken_counts:
        for pair in local_pair_counts(sequence):
            index[pair].add(sequence)

    return dict(index)

In [15]:
demo = Counter(
    {
        (b"a", b"b", b"c"): 3,
        (b"x", b"a", b"b"): 2,
        (b"d", b"e"): 5,
    }
)
demo_index = build_pair_index(demo)

print("(a,b) occurs in:", demo_index[(b"a", b"b")])

(a,b) occurs in: {(b'x', b'a', b'b'), (b'a', b'b', b'c')}


## 7. Incremental Update 的数学原理
假设一个 affected sequence 的 corpus frequency 为 $f$